In [45]:
"""
Sparse Retrieval (BM25) RAG Pipeline -- Production-Grade
=========================================================
Architecture   : FIVE BM25 configurations compared side by side:
                 (1) BM25Okapi  -- standard Okapi BM25 (default, Elasticsearch default)
                 (2) BM25Plus   -- eliminates short-document bias via delta floor
                 (3) BM25L      -- length-corrected variant for long-document corpora
                 (4) Custom preprocessing  -- stemming + stopword removal
                 (5) MultiField BM25       -- title + body field-weighted retrieval

Sparse Backend : rank-bm25 (BM25Okapi, BM25Plus, BM25L all built-in)
LLM            : Azure OpenAI  & (ChatOllama )
Data Source    : Three publicly available research PDFs from arXiv (open-access)
Framework      : LangChain BM25Retriever  (langchain-community)

Reference PDFs:
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf

    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf

    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
Core Concept -- WHAT IS SPARSE RETRIEVAL AND THE BM25 FAMILY
=============================================================================
Sparse retrieval represents text as high-dimensional but sparse vectors where
each dimension corresponds to a vocabulary token. "Sparse" means that for any
given document, nearly all dimensions are zero -- only the tokens that actually
appear in the document have non-zero values. This is in sharp contrast to dense
retrieval (all-MiniLM, BGE) where every dimension carries a real-valued signal.

BM25 (Best Match 25) is the canonical sparse retrieval function. It is the
industry default in every major search engine: Elasticsearch, OpenSearch,
Apache Solr, and Lucene all use BM25 as their default ranking function.
"Okapi BM25" refers to the variant developed at City University London for
the Okapi information retrieval system (Robertson et al., 1994).

THE BM25 SCORING FORMULA (Okapi variant):

    score(D, Q) = sum over terms t in Q:
        IDF(t) * [ TF(t, D) * (k1 + 1) ] / [ TF(t, D) + k1 * (1 - b + b * |D|/avgdl) ]

    where:
        TF(t, D)  = raw term frequency of term t in document D
        IDF(t)    = log( (N - df(t) + 0.5) / (df(t) + 0.5) + 1 )
                    N = total documents, df(t) = documents containing t
        |D|       = length of document D in tokens
        avgdl     = average document length across all documents
        k1        = term frequency saturation parameter  (default: 1.5)
        b         = document length normalization factor (default: 0.75)

The k1 and b parameters are the two fundamental tuning handles:
    k1 controls how much increasing TF continues to matter. At k1=0, TF is
    ignored entirely (pure IDF). At k1=inf, raw TF is used with no saturation.
    k1=1.5 means that for a term appearing 2 times vs. 1 time, the score
    increases by a factor of about 1.17 (not 2.0 -- saturating behavior).

    b controls length normalization. At b=0, document length has no effect.
    At b=1.0, scores are fully normalized to document length. b=0.75 is the
    standard empirically validated default across TREC benchmarks.

=============================================================================
Why BM25 Has Three Variants (Okapi, Plus, L)
=============================================================================
BM25Okapi has a known failure mode for very short documents: a single
matching term in a 3-token chunk scores extremely high because the length
normalization denominator is small. BM25Plus and BM25L were developed to
correct this without breaking performance on average-length documents.

BM25Plus (Lv & Zhai, 2011):
    Modifies the TF component by adding a floor value delta:
    TF_plus = delta + TF_normalized
    This ensures every matched term contributes AT LEAST delta to the score,
    eliminating zero-contribution matched terms for very short documents.
    Particularly effective for RAG where chunked documents vary greatly in length.

BM25L (Lv & Zhai, 2011):
    Modifies the TF normalization differently, capping the length penalty.
    Better for corpora with very long documents (books, legal contracts).
    Less common in RAG pipelines but included for completeness.

=============================================================================
Why Preprocessing Matters for BM25
=============================================================================
BM25 is a BAG-OF-WORDS model. It has NO understanding of morphology. Without
preprocessing:
    "run", "running", "runs", "ran" are FOUR DIFFERENT TOKENS.
    A query for "run" scores zero against a document saying "running".

With stemming (Porter Stemmer):
    "run", "running", "runs", "ran" all reduce to "run".
    The matching works correctly.

With stopword removal:
    "the", "a", "of", "and", "in" are removed before indexing.
    IDF scores for high-frequency function words are suppressed.
    Memory footprint of the BM25 index shrinks by ~30-40%.
    Retrieval precision improves because query tokens are all content words.


"""


'\nSparse Retrieval (BM25) RAG Pipeline -- Production-Grade\n=========================================================\nArchitecture   : FIVE BM25 configurations compared side by side:\n                 (1) BM25Okapi  -- standard Okapi BM25 (default, Elasticsearch default)\n                 (2) BM25Plus   -- eliminates short-document bias via delta floor\n                 (3) BM25L      -- length-corrected variant for long-document corpora\n                 (4) Custom preprocessing  -- stemming + stopword removal\n                 (5) MultiField BM25       -- title + body field-weighted retrieval\n\nSparse Backend : rank-bm25 (BM25Okapi, BM25Plus, BM25L all built-in)\nLLM            : Azure OpenAI  & (ChatOllama )\nData Source    : Three publicly available research PDFs from arXiv (open-access)\nFramework      : LangChain BM25Retriever  (langchain-community)\n\nReference PDFs:\n    1. "Attention Is All You Need" -- Vaswani et al., 2017\n       https://arxiv.org/pdf/1706.03762.pdf\n\n  

In [46]:
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords   

# Example usage
nltk.download('stopwords')  # make sure the corpus is downloaded
stop_words = set(stopwords.words('english'))

print("Number of stopwords:", len(stop_words))
print("First 10 stopwords:", list(stop_words)[:10])


Number of stopwords: 198
First 10 stopwords: ['is', 'have', "shouldn't", 're', 'again', 'ain', 'are', 'the', 'all', "it'd"]


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dhanu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [47]:
import os 
import re 
import sys 
import time 
import string 
import logging 
import pickle 
import urllib.request 
from pathlib import Path 
from typing import List ,Dict, Tuple , Callable , Optional

import nltk 
from nltk.stem import PorterStemmer 
from nltk.corpus import  stopwords  
NLTK_AVAILABLE = True 

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader

from langchain_classic.retrievers import BM25Retriever
from langchain_ollama import ChatOllama 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import ParentDocumentRetriever,ContextualCompressionRetriever,MergerRetriever,MultiVectorRetriever


In [48]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("sparse_bm25_rag")


In [49]:
class BM25RAGConfig:
    """
    Centralized configuration for the Sparse Retrieval BM25 RAG pipeline.

    BM25 Parameter Tuning Reference:
        The default parameters (k1=1.5, b=0.75) are the Elasticsearch defaults
        and have been validated extensively across TREC benchmarks. Deviation
        should be based on corpus-specific evaluation:

        k1 tuning:
            Short, uniform-length chunks (RAG, < 512 chars): k1=1.2 reduces
            TF saturation aggressively -- since chunks are short, term frequency
            rarely exceeds 3-4, so aggressive saturation is appropriate.
            Long documents (books, legal): k1=2.0 to maintain TF sensitivity.
            Academic papers with technical terminology: k1=1.5 (default).

        b tuning:
            Highly variable document lengths: b=0.75 (default) normalizes well.
            Fixed-size RAG chunks: b=0.3 to 0.5, since chunks are already
            length-normalized by the splitter. Full length normalization on
            fixed-size chunks over-penalizes chunks that are just slightly longer.
            Single-field, uniform text: b=0.5.

        BM25Plus delta:
            delta=0.5 is the recommended default from Lv & Zhai (2011).
            Higher delta (1.0) gives more weight to rare but matched terms.
            Lower delta (0.25) is closer to standard BM25 behavior.

    Chunk Size for BM25:
        BM25 performs best on chunks of 200-600 characters (roughly 30-90 words).
        At this size, individual chunks tend to be about one technical claim or
        one paragraph -- topically focused with clean TF profiles.
        Very small chunks (< 100 chars) create degenerate IDF scores because
        the corpus vocabulary collapses. Very large chunks (> 1000 chars) make
        BM25 behave like a full-document ranking function rather than a passage
        retrieval function.
    """
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # Text splitting 
    CHUNK_SIZE :int = 400 
    CHUNK_OVERLAP :int = 50

    # Config 1: BM25Okapi (standard)
    BM25_OKAPI_PARAMS: Dict = {"k1": 1.5, "b": 0.75, "epsilon": 0.25}
    # epsilon is BM25Okapi's floor value for IDF scores -- prevents negative IDF
    # for very common terms. Default 0.25 works well for technical text.

    # Config 2: BM25Plus
    BM25_PLUS_PARAMS: Dict = {"k1": 1.5, "b": 0.75, "delta": 0.5}
    # delta is the minimum contribution floor for matched terms.
    # 0.5 is the canonical value from the original BM25Plus paper.

    # Config 3: BM25L
    BM25_L_PARAMS: Dict = {"k1": 1.5, "b": 0.75, "delta": 0.5}
    # BM25L uses delta to cap the length normalization.

    # --- Top-K Results ---------------------------------------------------
    RETRIEVER_K: int = 4    # documents returned per query

    # --- BM25 Index Persistence ------------------------------------------
    BM25_INDEX_DIR: str = "./bm25_index"

    # --- Azure OpenAI LLM ------------------------------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_TEMPERATURE: float        = 0.0
    LLM_MAX_TOKENS: int           = 1024

    # --- RAG Generation Prompt -------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information provided in the context below.
If the answer is not present in the context, respond with:
"The provided documents do not contain sufficient information to answer this question."

Context (retrieved from research papers using sparse BM25 retrieval):
{context}

Question: {question}

Provide a technically accurate, concise, well-structured answer:"""



In [50]:

def default_preprocessing(text: str) -> List[str]:
    """
    Default BM25 preprocessing: lowercase and whitespace tokenization.

    This is the LangChain BM25Retriever default. It is the minimal viable
    preprocessing -- just lowercases and splits on whitespace. Punctuation
    attached to words (e.g., "model." or "paper,") remains as part of the token,
    which slightly degrades matching quality.

    Used in Configs 1, 2, and 3 to isolate the effect of the BM25 variant
    itself from the preprocessing choice.

    Args:
        text: Raw chunk text string.

    Returns:
        List of lowercase whitespace-split tokens.
    """
    return text.lower().split()


def clean_preprocessing(text: str) -> List[str]:
    """
    Improved preprocessing: lowercase + punctuation removal + whitespace tokenization.

    Removes all punctuation before tokenization so "model." and "model" are
    treated as the same token. Also removes standalone digits and single
    characters, which are rarely useful BM25 features in technical text.

    Without punctuation removal:
        "accuracy." != "accuracy"  -> missed match
        "attention," != "attention" -> missed match

    With punctuation removal:
        "attention." -> "attention" = "attention" -> match

    Args:
        text: Raw chunk text string.

    Returns:
        List of clean lowercase tokens.
    """
    # Lowercase
    text = text.lower()
    # Remove punctuation (preserves hyphens for compound terms like "pre-training")
    text = re.sub(r"[^\w\s-]", " ", text)
    # Remove standalone numbers and single characters (noise in formulas/citations)
    tokens = [
        t for t in text.split()
        if len(t) > 1 and not re.fullmatch(r"\d+", t)
    ]
    return tokens


def stemming_preprocessing(text: str) -> List[str]:
    """
    Advanced preprocessing: punctuation removal + stopword filtering + Porter stemming.

    This is the linguistically richest preprocessing option. Porter Stemmer
    reduces words to their morphological root:
        "running" -> "run"
        "training" -> "train"
        "pre-trained" -> "pre-train"
        "embeddings" -> "embed"

    Combined with stopword removal, this produces the cleanest vocabulary
    for BM25 and maximizes recall on vocabulary-mismatched queries.

    Trade-off: Stemming can over-merge distinct terms ("bank" for "banking"
    and "bank" the financial institution both stem to "bank"). For highly
    specialized technical vocabulary, this is rarely a problem.

    NLTK must be installed and punkt/stopwords must be downloaded:
        python -c "import nltk; nltk.download('punkt'); nltk.download('stopwords')"

    Falls back to clean_preprocessing if NLTK is unavailable.

    Args:
        text: Raw chunk text string.

    Returns:
        List of stemmed, stopword-filtered lowercase tokens.
    """
    if not NLTK_AVAILABLE:
        logger.warning("NLTK not available; falling back to clean_preprocessing.")
        return clean_preprocessing(text)

    stemmer   = PorterStemmer()
    stop_words = set(stopwords.words("english"))

    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)    # remove all punctuation

    tokens = [
        stemmer.stem(token)
        for token in text.split()
        if token not in stop_words and len(token) > 2
    ]
    return tokens


def technical_preprocessing(text: str) -> List[str]:
    """
    Domain-specific preprocessing tuned for machine learning research papers.

    Applies clean tokenization while preserving:
        - Hyphenated compound terms ("self-attention", "pre-training", "bi-directional")
        - Alphanumeric model names and version strings ("GPT-3", "BERT-base")
        - Acronyms and abbreviations ("NLP", "BLEU", "MLM", "NSP")

    Removes:
        - Standalone punctuation and brackets
        - LaTeX math notation fragments (\\frac, \\sum, etc.)
        - Citation markers ([1], [2], etc.)
        - Isolated numbers (page refs, equation numbers)

    This preprocessing is most effective for technical research paper corpora
    where compound terms and acronyms carry significant information content.

    Args:
        text: Raw chunk text string.

    Returns:
        List of cleaned tokens preserving technical vocabulary.
    """
    # Remove LaTeX commands and math fragments
    text = re.sub(r"\\[a-zA-Z]+\{[^}]*\}", " ", text)    # \frac{a}{b}
    text = re.sub(r"\\[a-zA-Z]+", " ", text)               # standalone \command

    # Remove citation markers like [1], [2,3], [Vaswani et al., 2017]
    text = re.sub(r"\[\d+(?:,\s*\d+)*\]", " ", text)
    text = re.sub(r"\[[A-Z][a-z]+.*?\d{4}\]", " ", text)

    # Lowercase
    text = text.lower()

    # Keep hyphens for compound terms, remove other punctuation
    text = re.sub(r"[^\w\s-]", " ", text)

    # Tokenize, keeping compound terms and acronyms, filtering noise
    tokens = []
    for token in text.split():
        # Keep: acronyms (all-caps), compound terms (hyphenated), normal words
        if len(token) <= 1:
            continue
        if re.fullmatch(r"\d+", token):      # pure number: skip
            continue
        if re.fullmatch(r"-+", token):       # standalone dashes: skip
            continue
        tokens.append(token)

    return tokens

In [51]:

# ===========================================================================
# SECTION 5 -- BM25 INDEX PERSISTENCE UTILITIES
# BM25Retriever is in-memory only in LangChain. This section adds
# pickle-based persistence so the index can be saved and reloaded
# across process restarts without rebuilding.
# ===========================================================================

def save_bm25_retriever(retriever: BM25Retriever, path: str) -> None:
    """
    Serialize a BM25Retriever to disk using pickle.

    BM25Retriever contains:
        - retriever.vectorizer : the rank_bm25.BM25Okapi/Plus/L object
          (stores IDF values, token counts, corpus statistics)
        - retriever.docs       : the original Document list with metadata
        - retriever.preprocess_func : the tokenization callable

    All three are picklable. The saved file is approximately:
        num_docs * avg_tokens_per_doc * 8 bytes (float64 IDF array)
    For 5000 chunks with avg 60 tokens: ~2.4 MB per saved index.

    SECURITY NOTE: Never load a pickle file from an untrusted source.
    For production, consider using a dedicated inverted index format
    (Whoosh, Lucene) instead of pickle for security and portability.

    Args:
        retriever : Populated BM25Retriever to serialize.
        path      : Output file path (e.g., "./bm25_index/okapi.pkl").
    """
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(
            {
                "vectorizer":      retriever.vectorizer,
                "docs":            retriever.docs,
                "k":               retriever.k,
                "preprocess_func": retriever.preprocess_func,
            },
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    size_kb = Path(path).stat().st_size / 1024
    logger.info("BM25 index saved: %s  (%.1f KB)", path, size_kb)


def load_bm25_retriever(path: str) -> Optional[BM25Retriever]:
    """
    Load a serialized BM25Retriever from disk.

    Reconstructs the BM25Retriever with the saved vectorizer, document list,
    k value, and preprocessing function. The loaded retriever is functionally
    identical to the one that was saved.

    Args:
        path: Path to the saved pickle file.

    Returns:
        Loaded BM25Retriever, or None if the file does not exist.
    """
    if not Path(path).exists():
        return None

    logger.info("Loading BM25 index from: %s", path)
    with open(path, "rb") as f:
        state = pickle.load(f)

    retriever = BM25Retriever(
        vectorizer=state["vectorizer"],
        docs=state["docs"],
        k=state["k"],
        preprocess_func=state["preprocess_func"],
    )
    logger.info("BM25 index loaded.  Corpus size: %d docs", len(retriever.docs))
    return retriever


# ===========================================================================
# SECTION 6 -- FIVE BM25 RETRIEVER CONFIGURATIONS
# ===========================================================================


In [60]:
def _build_bm25_retriever(
    chunks: List[Document],
    preprocess_func: Callable[[str], List[str]],
    bm25_params: Dict,
    variant: str = "okapi",
) -> BM25Retriever:
    """Build BM25 retriever with variant compatibility across package versions."""
    variant = (variant or "okapi").lower()

    if variant == "okapi":
        return BM25Retriever.from_documents(
            chunks,
            bm25_params=bm25_params,
            preprocess_func=preprocess_func,
        )

    # Newer rank_bm25 supports BM25Plus/BM25L; older langchain wrappers may not.
    try:
        from rank_bm25 import BM25L, BM25Okapi, BM25Plus

        variant_cls = {
            "okapi": BM25Okapi,
            "plus": BM25Plus,
            "l": BM25L,
        }.get(variant, BM25Okapi)

        tokenized_corpus = [preprocess_func(d.page_content) for d in chunks]
        vectorizer = variant_cls(tokenized_corpus, **bm25_params)
        return BM25Retriever(
            vectorizer=vectorizer,
            docs=chunks,
            preprocess_func=preprocess_func,
        )
    except Exception as exc:
        safe_okapi_params = {k: v for k, v in bm25_params.items() if k in {"k1", "b", "epsilon"}}
        logger.warning(
            "BM25 variant '%s' unsupported in current environment (%s). Falling back to BM25Okapi.",
            variant,
            exc,
        )
        return BM25Retriever.from_documents(
            chunks,
            bm25_params=safe_okapi_params,
            preprocess_func=preprocess_func,
        )


def build_config1_bm25_okapi(
    chunks: List[Document],
    config: BM25RAGConfig,
) -> BM25Retriever:
    """
    Configuration 1: BM25Okapi with default preprocessing.

    BM25Okapi is the exact implementation of Robertson et al.'s Okapi BM25
    as published. It is the universally accepted baseline for sparse retrieval
    and the starting point for evaluating all BM25 variants.

    epsilon=0.25 is the IDF floor value specific to BM25Okapi. It prevents
    negative IDF scores for terms that appear in more than half the corpus --
    a rare but possible situation with very common technical terms like
    "model" or "attention" in a research paper corpus.

    Parameters (k1=1.5, b=0.75):
        These are the Elasticsearch defaults, validated across TREC benchmarks.
        No tuning is applied here to establish a clean baseline.

    Index persistence:
        Builds from scratch on first call and saves to disk via pickle.
        Loads from disk on subsequent calls (skip rebuild).

    Args:
        chunks : All document chunks.
        config : BM25RAGConfig with BM25_OKAPI_PARAMS and RETRIEVER_K.

    Returns:
        Populated BM25Retriever using BM25Okapi variant.
    """
    cache_path = f"{config.BM25_INDEX_DIR}/okapi.pkl"
    cached     = load_bm25_retriever(cache_path)
    if cached:
        return cached

    logger.info("Building BM25Okapi retriever from %d chunks ...", len(chunks))
    retriever = _build_bm25_retriever(
        chunks,
        preprocess_func=default_preprocessing,
        bm25_params=config.BM25_OKAPI_PARAMS,
        variant="okapi",
    )
    retriever.k = config.RETRIEVER_K
    save_bm25_retriever(retriever, cache_path)
    return retriever

In [61]:
def build_config2_bm25_plus(
    chunks: List[Document],
    config: BM25RAGConfig,
) -> BM25Retriever:
    """Configuration 2: BM25Plus with default preprocessing."""
    cache_path = f"{config.BM25_INDEX_DIR}/plus.pkl"
    cached = load_bm25_retriever(cache_path)
    if cached:
        return cached

    logger.info("Building BM25Plus retriever from %d chunks ...", len(chunks))
    retriever = _build_bm25_retriever(
        chunks,
        preprocess_func=default_preprocessing,
        bm25_params=config.BM25_PLUS_PARAMS,
        variant="plus",
    )
    retriever.k = config.RETRIEVER_K
    save_bm25_retriever(retriever, cache_path)
    return retriever

In [62]:
def build_config3_bm25_l(
    chunks: List[Document],
    config: BM25RAGConfig,
) -> BM25Retriever:
    """Configuration 3: BM25L with default preprocessing."""
    cache_path = f"{config.BM25_INDEX_DIR}/l.pkl"
    cached = load_bm25_retriever(cache_path)
    if cached:
        return cached

    logger.info("Building BM25L retriever from %d chunks ...", len(chunks))
    retriever = _build_bm25_retriever(
        chunks,
        preprocess_func=default_preprocessing,
        bm25_params=config.BM25_L_PARAMS,
        variant="l",
    )
    retriever.k = config.RETRIEVER_K
    save_bm25_retriever(retriever, cache_path)
    return retriever

In [63]:
def build_config4_stemmed_bm25(
    chunks: List[Document],
    config: BM25RAGConfig,
) -> BM25Retriever:
    """Configuration 4: BM25Plus + stemming + stopword preprocessing."""
    cache_path = f"{config.BM25_INDEX_DIR}/stemmed_plus.pkl"
    cached = load_bm25_retriever(cache_path)
    if cached:
        return cached

    if NLTK_AVAILABLE:
        for resource in ["punkt", "stopwords"]:
            try:
                nltk.data.find(f"tokenizers/{resource}")
            except LookupError:
                logger.info("Downloading NLTK resource: %s", resource)
                nltk.download(resource, quiet=True)

    logger.info("Building BM25Plus + Stemming retriever from %d chunks ...", len(chunks))
    retriever = _build_bm25_retriever(
        chunks,
        preprocess_func=stemming_preprocessing,
        bm25_params=config.BM25_PLUS_PARAMS,
        variant="plus",
    )
    retriever.k = config.RETRIEVER_K
    save_bm25_retriever(retriever, cache_path)
    return retriever

In [64]:
def build_config5_technical_bm25(
    chunks: List[Document],
    config: BM25RAGConfig,
) -> BM25Retriever:
    """Configuration 5: BM25Plus + domain-specific technical preprocessing."""
    cache_path = f"{config.BM25_INDEX_DIR}/technical_plus.pkl"
    cached = load_bm25_retriever(cache_path)
    if cached:
        return cached

    logger.info(
        "Building BM25Plus + Technical preprocessing retriever from %d chunks ...",
        len(chunks),
    )
    retriever = _build_bm25_retriever(
        chunks,
        preprocess_func=technical_preprocessing,
        bm25_params=config.BM25_PLUS_PARAMS,
        variant="plus",
    )
    retriever.k = config.RETRIEVER_K
    save_bm25_retriever(retriever, cache_path)
    return retriever


# ===========================================================================
# SECTION 7 -- BM25 INDEX DIAGNOSTICS
# Inspect the IDF scores and corpus vocabulary after building an index.
# These diagnostics reveal the quality of the index and help identify
# preprocessing issues before running production queries.
# ===========================================================================


In [57]:
def inspect_bm25_index(
    retriever: BM25Retriever,
    label: str,
    top_n_idf: int = 20,
) -> None:
    """
    Print diagnostic statistics for a built BM25 index.

    Works across rank-bm25 variants where IDF may be exposed as either:
        - vectorizer.idf (dict[str, float])
        - vectorizer.idf_ (array/list)

    Args:
        retriever  : Populated BM25Retriever to inspect.
        label      : Human-readable label for display.
        top_n_idf  : Number of highest-IDF terms to display.
    """
    vectorizer  = retriever.vectorizer
    corpus_size = len(retriever.docs)

    # Rank-bm25 compatibility across versions
    idf_dict = {}
    if hasattr(vectorizer, "idf") and isinstance(getattr(vectorizer, "idf"), dict):
        idf_dict = vectorizer.idf
    elif hasattr(vectorizer, "idf_"):
        idf_values = list(getattr(vectorizer, "idf_"))
        corpus_tokens = getattr(vectorizer, "corpus", [])
        unique_terms = []
        seen = set()
        for doc_tokens in corpus_tokens:
            for tok in doc_tokens:
                if tok not in seen:
                    seen.add(tok)
                    unique_terms.append(tok)
        idf_dict = dict(zip(unique_terms[:len(idf_values)], idf_values))

    vocab_size = len(idf_dict)

    # Average document length in tokens
    avg_dl = float(getattr(vectorizer, "avgdl", 0.0))

    print(f"\n{'='*60}")
    print(f"BM25 INDEX DIAGNOSTICS: {label}")
    print(f"{'='*60}")
    print(f"  Corpus documents  : {corpus_size:,}")
    print(f"  Vocabulary size   : {vocab_size:,}")
    print(f"  Avg doc length    : {avg_dl:.1f} tokens")
    print(f"  Avg chunk chars   : {sum(len(d.page_content) for d in retriever.docs) / max(corpus_size, 1):.0f}")

    sample_doc_tokens = vectorizer.corpus[0][:15] if hasattr(vectorizer, "corpus") and vectorizer.corpus else []
    if sample_doc_tokens:
        print(f"\n  Sample tokens (chunk 0): {sample_doc_tokens}")

    if idf_dict:
        top_terms = sorted(idf_dict.items(), key=lambda x: x[1], reverse=True)[:top_n_idf]
        print(f"\n  Top {len(top_terms)} IDF terms:")
        for term, score in top_terms:
            print(f"    {term:<24} {score:.4f}")

    print(f"{'='*60}\n")

In [29]:
def build_ollama_llm(config: BM25RAGConfig) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) 
    temperature = getattr(config, 'LLM_TEMPERATURE', 0.0)
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', temperature),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )

In [31]:

def build_rag_chain(
    retriever: BM25Retriever,
    llm: ChatOllama,
    config: BM25RAGConfig,
    label: str = "",
):
    """
    Build a complete LCEL RAG chain wrapping any BM25Retriever configuration.

    Because all five BM25 configurations expose the same standard
    LangChain Runnable retriever interface, this single factory produces
    a valid RAG chain for all of them. The chain is:

        question -> retriever.invoke(question)
                 -> format_docs(docs)
                 -> prompt(context, question)
                 -> llm(prompt)
                 -> StrOutputParser()
                 -> answer string

    The format_docs function includes the BM25 score in the chunk header
    when available (BM25Retriever does NOT inject scores into metadata --
    this is a known LangChain limitation; scores are internal to rank_bm25).
    Instead, we label each chunk with its rank position (1, 2, 3, 4).

    Args:
        retriever : Any populated BM25Retriever.
        llm       : ChatOllama instance.
        config    : BM25RAGConfig with RAG_PROMPT.
        label     : Human-readable label for logging.

    Returns:
        Tuple of (LCEL rag_chain, retriever).
    """
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)

    def format_docs(docs: List[Document]) -> str:
        """
        Format BM25-retrieved documents into an annotated context block.

        Chunks are labeled with rank position (BM25 returns in ranked order),
        source paper, page number, and character length. The rank position
        is the most important label: BM25 chunk at rank 1 has the highest
        BM25 score and is presumed most relevant to the query.
        """
        parts = []
        for i, doc in enumerate(docs, start=1):
            paper = doc.metadata.get("paper_name", "Unknown")
            page  = doc.metadata.get("page", "?")
            chars = len(doc.page_content)
            parts.append(
                f"[BM25 Rank {i} | {paper} | Page {page} | {chars} chars]\n"
                f"{doc.page_content.strip()}"
            )
        return ("\n\n" + "-" * 60 + "\n\n").join(parts)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )

    logger.info("RAG chain ready: %s", label or "BM25")
    return chain, retriever

In [32]:

def query_all_bm25_configs(
    config_chains: Dict[str, tuple],
    question: str,
    show_retrieval_details: bool = True,
) -> Dict[str, Dict]:
    """
    Execute a question through all five BM25 configurations with timing.

    BM25 retrieval is extremely fast (< 5ms for 5000 docs) because it runs
    entirely in Python/NumPy with no model inference. The dominant cost is
    the LLM generation step (800-1200ms for GPT-4o). This makes BM25 the
    lowest-latency retrieval option in the RAG stack, with the caveat that
    it fails on vocabulary-mismatched queries.

    Displays:
        - Retrieved chunks with rank position, paper, page, character count
        - Preview of each chunk's content
        - LLM-generated answer
        - Comparison table of retrieval and generation latencies

    Args:
        config_chains          : Dict mapping label to (chain, retriever).
        question               : Natural language query.
        show_retrieval_details : If True, print chunk metadata.

    Returns:
        Dict mapping label to {"answer", "retrieval_ms", "generation_ms", "n_docs"}.
    """
    results: Dict[str, Dict] = {}

    print("\n" + "#" * 76)
    print(f"QUESTION: {question}")
    print("#" * 76)

    for label, (chain, retriever) in config_chains.items():
        print(f"\n{'='*60}")
        print(f"CONFIG: {label}")
        print("=" * 60)

        # Timed retrieval
        t0    = time.perf_counter()
        docs  = retriever.invoke(question)
        retr_ms = (time.perf_counter() - t0) * 1000

        if show_retrieval_details:
            print(f"\nBM25 Retrieved {len(docs)} chunks  (retrieval: {retr_ms:.1f}ms)")
            for i, doc in enumerate(docs, 1):
                paper = doc.metadata.get("paper_name", "?")
                page  = doc.metadata.get("page", "?")
                chars = len(doc.page_content)
                snip  = doc.page_content[:220].replace("\n", " ").strip()
                print(f"\n  [Rank {i}] {paper} | Page {page} | {chars} chars")
                print(f"           {snip}...")

        # Timed generation
        t1     = time.perf_counter()
        answer = chain.invoke(question)
        gen_ms = (time.perf_counter() - t1) * 1000

        print(f"\nANSWER (gen: {gen_ms:.0f}ms):\n{answer}")

        results[label] = {
            "answer":        answer,
            "retrieval_ms":  round(retr_ms, 1),
            "generation_ms": round(gen_ms, 1),
            "n_docs":        len(docs),
        }

    # Summary comparison table
    print("\n" + "=" * 76)
    print("LATENCY SUMMARY ACROSS ALL BM25 CONFIGURATIONS")
    print(f"{'Configuration':<42} {'Docs':>5} {'Retr(ms)':>10} {'Gen(ms)':>10}")
    print("-" * 76)
    for lbl, r in results.items():
        print(
            f"{lbl:<42} {r['n_docs']:>5} "
            f"{r['retrieval_ms']:>10.1f} {r['generation_ms']:>10.0f}"
        )
    print("=" * 76 + "\n")

    return results


In [33]:

def bm25_parameter_sweep(
    chunks: List[Document],
    query: str,
    k1_values: List[float] = [0.5, 1.0, 1.5, 2.0],
    b_values:  List[float] = [0.25, 0.5, 0.75, 1.0],
    top_k: int = 3,
) -> None:
    """
    Sweep k1 and b parameter combinations and display top-k retrieved chunks
    for each combination.

    This diagnostic function is the standard production approach for BM25
    parameter tuning. For each (k1, b) pair, it builds a temporary BM25Okapi
    retriever, retrieves the top-k chunks for the query, and prints the
    source paper + page of each retrieved chunk. You observe how the retrieved
    set changes as parameters vary and select the configuration that retrieves
    the most relevant chunks for representative queries from your corpus.

    Automated tuning: In a production scenario with a labeled evaluation set
    (query + known relevant document pairs), replace the print output with a
    Recall@K or NDCG@K computation and select the (k1, b) pair maximizing
    the metric. This is the standard approach used in Elasticsearch tuning.

    Args:
        chunks     : Document chunks to index for the sweep.
        query      : Test query to evaluate.
        k1_values  : List of k1 values to try.
        b_values   : List of b values to try.
        top_k      : Number of top results to display per parameter combination.
    """
    print("\n" + "=" * 70)
    print(f"BM25 PARAMETER SWEEP")
    print(f"Query: '{query}'")
    print(f"{'k1':>6} {'b':>6} {'Rank 1 Source':>30} {'Rank 2 Source':>30}")
    print("-" * 70)

    for k1 in k1_values:
        for b in b_values:
            # Build temporary BM25Okapi retriever with these parameters
            ret = BM25Retriever.from_documents(
                chunks,
                bm25_params={"k1": k1, "b": b},
                preprocess_func=default_preprocessing,
            )
            ret.k = top_k
            docs = ret.invoke(query)

            sources = [
                f"{d.metadata.get('paper_name','?')[:28]:28} p{d.metadata.get('page','?')}"
                for d in docs[:2]
            ]
            while len(sources) < 2:
                sources.append("(no result)")

            print(f"{k1:>6.1f} {b:>6.2f}   {sources[0]:<30} {sources[1]:<30}")

    print("=" * 70 + "\n")



In [34]:
config = BM25RAGConfig()


In [35]:
llm = build_ollama_llm(config)

In [54]:
def load_and_chunk_documents(
    pdf_paths,
    config: BM25RAGConfig,
) -> List[Document]:
    """
    Load PDF pages and split into BM25-optimized chunks.

    Accepts either:
        1) A directory path containing PDFs
        2) A single PDF path
        3) A list of PDF paths

    Args:
        pdf_paths : Directory path, file path, or iterable of PDF paths.
        config    : BM25RAGConfig with CHUNK_SIZE and CHUNK_OVERLAP.

    Returns:
        List of chunked Documents with paper_name and source metadata.
    """
    # Normalize input into a list[Path] so callers can pass either a folder or paths.
    if isinstance(pdf_paths, (str, Path)):
        candidate = Path(pdf_paths)
        if candidate.is_dir():
            normalized_pdf_paths = sorted(candidate.glob("*.pdf"))
        else:
            normalized_pdf_paths = [candidate]
    else:
        normalized_pdf_paths = [Path(p) for p in pdf_paths]

    if not normalized_pdf_paths:
        raise RuntimeError("No PDF files found. Provide valid PDF path(s) or a directory with .pdf files.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )

    all_chunks: List[Document] = []

    for pdf_path in normalized_pdf_paths:
        if not pdf_path.exists():
            raise RuntimeError(f"PDF path not found: {pdf_path}")

        paper_name = pdf_path.stem.replace("_", " ").title()
        logger.info("Loading: %s", pdf_path.name)

        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        for page in pages:
            page.metadata["source"]     = pdf_path.name
            page.metadata["paper_name"] = paper_name

        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)

    avg_len = sum(len(c.page_content) for c in all_chunks) / max(len(all_chunks), 1)
    logger.info(
        "Total chunks: %d  |  avg %.0f chars  |  avg ~%.0f words",
        len(all_chunks), avg_len, avg_len / 5.5,
    )
    return all_chunks

In [55]:
chunks = load_and_chunk_documents(config.PDF_DOWNLOAD_DIR, config)

2026-05-18 22:00:31 | INFO     | sparse_bm25_rag | Loading: attention_is_all_you_need.pdf
2026-05-18 22:00:32 | INFO     | sparse_bm25_rag |   attention_is_all_you_need.pdf -> 15 pages -> 119 chunks
2026-05-18 22:00:32 | INFO     | sparse_bm25_rag | Loading: bert_pretraining_transformers.pdf
2026-05-18 22:00:35 | INFO     | sparse_bm25_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 189 chunks
2026-05-18 22:00:35 | INFO     | sparse_bm25_rag | Loading: rag_knowledge_intensive_nlp.pdf
2026-05-18 22:00:36 | INFO     | sparse_bm25_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 200 chunks
2026-05-18 22:00:36 | INFO     | sparse_bm25_rag | Total chunks: 508  |  avg 354 chars  |  avg ~64 words


In [58]:
r1 = build_config1_bm25_okapi(chunks, config)
inspect_bm25_index(r1, "Config 1: BM25Okapi + Default Preprocessing")

2026-05-18 22:01:17 | INFO     | sparse_bm25_rag | Loading BM25 index from: ./bm25_index/okapi.pkl
2026-05-18 22:01:17 | INFO     | sparse_bm25_rag | BM25 index loaded.  Corpus size: 508 docs

BM25 INDEX DIAGNOSTICS: Config 1: BM25Okapi + Default Preprocessing
  Corpus documents  : 508
  Vocabulary size   : 6,719
  Avg doc length    : 53.8 tokens
  Avg chunk chars   : 354

  Top 20 IDF terms:
    proper                   5.8240
    attribution              5.8240
    provided,                5.8240
    hereby                   5.8240
    grants                   5.8240
    permission               5.8240
    reproduce                5.8240
    tables                   5.8240
    figures                  5.8240
    journalistic             5.8240
    scholarly                5.8240
    vaswani∗                 5.8240
    avaswani@google.com      5.8240
    shazeer∗                 5.8240
    noam@google.com          5.8240
    parmar∗                  5.8240
    nikip@google.com        

In [65]:
r2 = build_config2_bm25_plus(chunks, config)


2026-05-18 22:07:18 | INFO     | sparse_bm25_rag | Building BM25Plus retriever from 508 chunks ...
2026-05-18 22:07:18 | INFO     | sparse_bm25_rag | BM25 index saved: ./bm25_index/plus.pkl  (636.1 KB)


In [66]:
r3 = build_config3_bm25_l(chunks, config)


2026-05-18 22:07:22 | INFO     | sparse_bm25_rag | Building BM25L retriever from 508 chunks ...
2026-05-18 22:07:22 | INFO     | sparse_bm25_rag | BM25 index saved: ./bm25_index/l.pkl  (636.1 KB)


In [67]:
r4 = build_config4_stemmed_bm25(chunks, config)


2026-05-18 22:07:26 | INFO     | sparse_bm25_rag | Downloading NLTK resource: stopwords
2026-05-18 22:07:26 | INFO     | sparse_bm25_rag | Building BM25Plus + Stemming retriever from 508 chunks ...
2026-05-18 22:07:27 | INFO     | sparse_bm25_rag | BM25 index saved: ./bm25_index/stemmed_plus.pkl  (499.8 KB)


In [68]:
r5 = build_config5_technical_bm25(chunks, config)


2026-05-18 22:07:44 | INFO     | sparse_bm25_rag | Building BM25Plus + Technical preprocessing retriever from 508 chunks ...
2026-05-18 22:07:44 | INFO     | sparse_bm25_rag | BM25 index saved: ./bm25_index/technical_plus.pkl  (571.9 KB)


In [69]:
inspect_bm25_index(r5, "Config 5: BM25Plus + Technical")



BM25 INDEX DIAGNOSTICS: Config 5: BM25Plus + Technical
  Corpus documents  : 508
  Vocabulary size   : 4,343
  Avg doc length    : 48.6 tokens
  Avg chunk chars   : 354

  Top 20 IDF terms:
    proper                   6.2324
    attribution              6.2324
    hereby                   6.2324
    grants                   6.2324
    permission               6.2324
    reproduce                6.2324
    figures                  6.2324
    journalistic             6.2324
    scholarly                6.2324
    avaswani                 6.2324
    nikip                    6.2324
    toronto                  6.2324
    lukaszkaiser             6.2324
    gmail                    6.2324
    dominant                 6.2324
    dispensing               6.2324
    superior                 6.2324
    parallelizable           6.2324
    english-                 6.2324
    to-german                6.2324



In [70]:
bm25_parameter_sweep(
        chunks,
        query="scaled dot-product attention mechanism",
        k1_values=[0.5, 1.0, 1.5, 2.0],
        b_values=[0.25, 0.5, 0.75],
        top_k=3,
    )
    # Step 6: Parameter sweep for a representative query



BM25 PARAMETER SWEEP
Query: 'scaled dot-product attention mechanism'
    k1      b                  Rank 1 Source                  Rank 2 Source
----------------------------------------------------------------------
   0.5   0.25   Attention Is All You Need    p3 Attention Is All You Need    p3
   0.5   0.50   Attention Is All You Need    p3 Attention Is All You Need    p3
   0.5   0.75   Attention Is All You Need    p3 Attention Is All You Need    p3
   1.0   0.25   Attention Is All You Need    p3 Attention Is All You Need    p3
   1.0   0.50   Attention Is All You Need    p3 Attention Is All You Need    p3
   1.0   0.75   Attention Is All You Need    p3 Attention Is All You Need    p3
   1.5   0.25   Attention Is All You Need    p3 Attention Is All You Need    p3
   1.5   0.50   Attention Is All You Need    p3 Attention Is All You Need    p3
   1.5   0.75   Attention Is All You Need    p3 Attention Is All You Need    p3
   2.0   0.25   Attention Is All You Need    p3 Attention Is Al

In [71]:
# Step 7: Build LCEL RAG chains
chain1, _ = build_rag_chain(r1, llm, config, "Config 1: BM25Okapi (default)")
chain2, _ = build_rag_chain(r2, llm, config, "Config 2: BM25Plus  (delta=0.5)")
chain3, _ = build_rag_chain(r3, llm, config, "Config 3: BM25L     (delta=0.5)")
chain4, _ = build_rag_chain(r4, llm, config, "Config 4: BM25Plus + Stemming")
chain5, _ = build_rag_chain(r5, llm, config, "Config 5: BM25Plus + Technical [BEST]")

config_chains: Dict[str, tuple] = {
    "Config 1: BM25Okapi (default)":       (chain1, r1),
    "Config 2: BM25Plus  (delta=0.5)":     (chain2, r2),
    "Config 3: BM25L     (delta=0.5)":     (chain3, r3),
    "Config 4: BM25Plus + Stemming":       (chain4, r4),
    "Config 5: BM25Plus + Technical [BEST]":(chain5, r5),
}

2026-05-18 22:12:49 | INFO     | sparse_bm25_rag | RAG chain ready: Config 1: BM25Okapi (default)
2026-05-18 22:12:49 | INFO     | sparse_bm25_rag | RAG chain ready: Config 2: BM25Plus  (delta=0.5)
2026-05-18 22:12:49 | INFO     | sparse_bm25_rag | RAG chain ready: Config 3: BM25L     (delta=0.5)
2026-05-18 22:12:49 | INFO     | sparse_bm25_rag | RAG chain ready: Config 4: BM25Plus + Stemming
2026-05-18 22:12:49 | INFO     | sparse_bm25_rag | RAG chain ready: Config 5: BM25Plus + Technical [BEST]


In [72]:
# Step 8: Demo queries
# Each query is designed to stress-test a different BM25 characteristic:
demo_questions = [
    # Exact keyword match -- all BM25 variants should perform similarly
    "What is the BLEU score of the Transformer model on WMT 2014 English-to-German?",

    # Stemming benefit -- "computing" vs "computation" vocabulary mismatch
    "How is the attention score computation performed in multi-head attention?",

    # Technical compound term -- benefits from domain preprocessing
    "What is the role of self-attention in the encoder-decoder architecture?",

    # BM25Plus advantage -- query term appears in a short section header chunk
    "What pre-training tasks are used in BERT?",

    # Cross-paper query -- tests IDF discrimination between papers
    "How does the RAG model retrieve relevant documents during inference?",
]

In [73]:
logger.info(
        "Running %d demo queries across all 5 BM25 configurations ...",
        len(demo_questions),
    )


2026-05-18 22:12:52 | INFO     | sparse_bm25_rag | Running 5 demo queries across all 5 BM25 configurations ...


In [76]:
for question in demo_questions:
        query_all_bm25_configs(config_chains, question, show_retrieval_details=True)




############################################################################
QUESTION: What is the BLEU score of the Transformer model on WMT 2014 English-to-German?
############################################################################

CONFIG: Config 1: BM25Okapi (default)

BM25 Retrieved 4 chunks  (retrieval: 3.1ms)

  [Rank 1] Attention Is All You Need | Page 0 | 398 chars
           be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best re...

  [Rank 2] Attention Is All You Need | Page 7 | 361 chars
           the competitive models. On the WMT 2014 English-to-French translation task, our big model achieves a BLEU score of 41.0, outperforming all of the previously published single models, at less than 1/4 the training cost of...

  [Rank 3] Attention Is All You Need | Page 7 | 330 chars
           Pdrop = 0.1. La